# Bagian A — Setup Data & Profil Dataset

**PJ:** Anggota A

Isi: load 4 CSV, parse `Date`, gabung jadi `df_all`, pivot `adj_wide` & `vol_wide`, `WARNA`, style plot, profil dataset.

## Setup Bersama

> ⚠️ **Sel di bawah ini adalah SETUP BERSAMA (dibuat Bagian A).** Jangan diubah di notebook masing-masing. Kalau perlu perubahan, minta Anggota A mengubahnya di `kerja/A_setup_data.ipynb`, lalu semua anggota copy ulang.

Variabel yang tersedia setelah sel ini dijalankan:

| Variabel | Isi |
|---|---|
| `TICKERS` | `['BBCA', 'BBRI', 'BMRI', 'BBNI']` (urutan tetap) |
| `NAMA_BANK` | nama lengkap tiap bank |
| `WARNA` | warna tetap per bank (selalu pakai ini, jangan pilih warna sendiri) |
| `df_all` | long format: `Date, Ticker, Open, High, Low, Close, Adj Close, Volume` |
| `adj_wide` | pivot **Adj Close** (index `Date`, kolom = ticker) |
| `vol_wide` | pivot **Volume** (index `Date`, kolom = ticker) |
| `HARI_DIBUANG` | tanggal placeholder hari libur (Volume 0, harga datar) yang dibuang |
| `FIG_DIR` | folder `figures/` untuk menyimpan gambar |
| `simpan_fig(fig, nama)` | simpan gambar ke `figures/<nama>.png` |
| `format_tanggal(ax)` | rapikan sumbu-x tanggal (label per tahun) |

Notebook bisa dijalankan dari root repo, dari folder `kerja/`, atau dari Google Colab (data otomatis diambil dari GitHub).

In [ ]:
# ===== SETUP BERSAMA (Bagian A) - JANGAN DIUBAH, copy persis ke semua notebook =====
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

TICKERS = ['BBCA', 'BBRI', 'BMRI', 'BBNI']
NAMA_BANK = {
    'BBCA': 'Bank Central Asia',
    'BBRI': 'Bank Rakyat Indonesia',
    'BMRI': 'Bank Mandiri',
    'BBNI': 'Bank Negara Indonesia',
}
# Palet kategorikal tervalidasi (aman untuk buta warna pada grafik garis/stacked).
# Warna mengikuti bank, bukan urutan ranking - jangan ditukar antar plot.
WARNA = {
    'BBCA': '#2a78d6',  # biru
    'BBRI': '#eb6834',  # oranye
    'BMRI': '#1baf7a',  # aqua
    'BBNI': '#eda100',  # kuning
}

SUMBER_DATA = 'https://www.kaggle.com/datasets/cmglonly/dataset-saham-bank-indonesia-2014-2024/data'
GITHUB_RAW = 'https://raw.githubusercontent.com/aprilianza/Midterm_DataVisualization/main'

# Cari root project: bisa dijalankan dari root repo maupun dari folder kerja/.
# Jika tidak ketemu (mis. di Colab), data diambil langsung dari GitHub.
ROOT = next((p for p in [Path.cwd(), Path.cwd().parent] if (p / 'dataset').is_dir()), None)
FIG_DIR = (ROOT or Path.cwd()) / 'figures'
FIG_DIR.mkdir(exist_ok=True)


def load_saham(ticker):
    nama_file = f'{ticker}.JK_10_years.csv'
    sumber = ROOT / 'dataset' / nama_file if ROOT else f'{GITHUB_RAW}/dataset/{nama_file}'
    df = pd.read_csv(sumber)
    # Kolom Date bertimezone UTC (00:00+00:00) -> jadikan tanggal biasa tanpa timezone
    df['Date'] = pd.to_datetime(df['Date'], utc=True).dt.tz_localize(None).dt.normalize()
    df['Ticker'] = ticker
    return df.sort_values('Date')


df_all = (
    pd.concat([load_saham(t) for t in TICKERS], ignore_index=True)
    .drop_duplicates(subset=['Ticker', 'Date'])
    [['Date', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']]
    .reset_index(drop=True)
)
# Baris placeholder hari libur bursa (Volume 0 & harga datar O=H=L=C) dari Yahoo Finance
# dibuang untuk SEMUA bank agar tanggal di adj_wide/vol_wide tetap sejajar.
_placeholder = (df_all['Volume'] == 0) & df_all[['Open', 'High', 'Low']].eq(df_all['Close'], axis=0).all(axis=1)
HARI_DIBUANG = pd.DatetimeIndex(sorted(df_all.loc[_placeholder, 'Date'].unique()))
df_all = df_all[~df_all['Date'].isin(HARI_DIBUANG)].reset_index(drop=True)

adj_wide = df_all.pivot(index='Date', columns='Ticker', values='Adj Close')[TICKERS]
vol_wide = df_all.pivot(index='Date', columns='Ticker', values='Volume')[TICKERS]

# Style plot seragam untuk semua anggota
plt.rcParams.update({
    'figure.figsize': (12, 5),
    'figure.dpi': 110,
    'savefig.dpi': 200,
    'savefig.bbox': 'tight',
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'axes.titlelocation': 'left',
    'axes.labelcolor': '#52514e',
    'axes.edgecolor': '#b5b4ae',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'axes.axisbelow': True,
    'grid.color': '#e6e5e0',
    'grid.linewidth': 0.8,
    'xtick.color': '#52514e',
    'ytick.color': '#52514e',
    'lines.linewidth': 1.6,
    'legend.frameon': False,
})


def simpan_fig(fig, nama):
    """Simpan figure ke figures/<nama>.png, mis. simpan_fig(fig, '01_line_bbca')."""
    path = FIG_DIR / f'{nama}.png'
    fig.savefig(path)
    print(f'Tersimpan: {path}')


def format_tanggal(ax):
    """Sumbu-x tanggal dengan label per tahun."""
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))


print(f'Data dimuat dari: {ROOT / "dataset" if ROOT else GITHUB_RAW}')
print(f'df_all: {df_all.shape[0]:,} baris | {df_all["Date"].min():%d %b %Y} - {df_all["Date"].max():%d %b %Y}')
print(f'adj_wide & vol_wide: {adj_wide.shape[0]:,} tanggal x {adj_wide.shape[1]} bank')
print(f'Hari placeholder (libur, Volume 0) yang dibuang: {len(HARI_DIBUANG)} tanggal')

## Profil Dataset

**Sumber:** Dataset Saham Bank Indonesia 2014–2024 (Kaggle) — https://www.kaggle.com/datasets/cmglonly/dataset-saham-bank-indonesia-2014-2024/data

Data asli diambil dari Yahoo Finance (ticker `.JK` = Bursa Efek Indonesia).

### Jumlah observasi & rentang waktu per bank

In [ ]:
ringkas = df_all.groupby('Ticker', sort=False).agg(
    Nama=('Ticker', lambda s: NAMA_BANK[s.iloc[0]]),
    Observasi=('Date', 'size'),
    Mulai=('Date', 'min'),
    Selesai=('Date', 'max'),
)
ringkas

### Missing value, duplikat, dan hari placeholder

Data mentah berisi baris dengan **Volume = 0 dan harga datar (Open = High = Low = Close)**. Tanggalnya jatuh pada hari libur bursa (17 Agustus, Natal, Tahun Baru, cuti bersama Lebaran), jadi baris ini bukan hari perdagangan sungguhan. Setup bersama **membuang** tanggal tersebut (`HARI_DIBUANG`) untuk semua bank agar tidak muncul return 0 dan lonjakan turun volume yang palsu.

In [ ]:
print(f'{len(HARI_DIBUANG)} tanggal dibuang, contoh:')
print(HARI_DIBUANG[:10].strftime('%d %b %Y (%a)').tolist())

In [ ]:
print('Missing value per kolom:')
print(df_all.isna().sum(), end='\n\n')
print('Duplikat (Ticker, Date):', df_all.duplicated(['Ticker', 'Date']).sum())
print('Tanggal tidak lengkap di 4 bank:', adj_wide.isna().any(axis=1).sum())
print('Sisa hari dengan Volume = 0 per bank:')
print((vol_wide == 0).sum())

### Granularitas waktu

Data **harian, hanya hari bursa** (tanpa Sabtu–Minggu dan hari libur). Tahun 2014 dan 2024 tidak penuh.

In [ ]:
hari_bursa = adj_wide.groupby(adj_wide.index.year).size().rename('Hari bursa')
hari_bursa.to_frame().T

In [ ]:
tgl = adj_wide.index.to_series()
selisih = tgl.diff().dt.days
print('Distribusi jarak antar observasi (hari):')
print(selisih.value_counts().sort_index().head(10), end='\n\n')
print('5 jeda terpanjang (libur panjang, mis. Lebaran):')
pd.DataFrame({'Dari': tgl.shift(), 'Sampai': tgl, 'Hari': selisih}).nlargest(5, 'Hari').reset_index(drop=True)

### Statistik deskriptif

In [ ]:
adj_wide.describe().round(0)

In [ ]:
(vol_wide.describe().drop('count') / 1e6).round(1).rename_axis('Volume (juta lembar)')

### Close vs Adj Close

- **Close**: harga penutupan apa adanya pada hari itu.
- **Adj Close**: harga yang sudah disesuaikan dengan **dividen** dan **stock split**, sehingga perubahan harga antar waktu bisa dibandingkan secara adil.

➡️ **Kesepakatan kelompok:** analisis tren dan perbandingan antar bank memakai **`Adj Close`** (`adj_wide`). `Close` hanya dipakai jika ingin menunjukkan harga nominal di pasar.

In [ ]:
rasio = (df_all['Adj Close'] / df_all['Close']).groupby(df_all['Ticker'], sort=False)
pd.DataFrame({'Rasio Adj/Close awal': rasio.first(), 'Rasio Adj/Close akhir': rasio.last()}).round(3)

### Insight Profil Dataset

**Insight:** _(diisi Anggota A — 2–3 kalimat tentang kualitas & karakteristik data)_